In [ ]:
# ============================================================
# COMP2090SEF - Property & Mortgage Calculator (Advanced)
# Features: Real 2026 Stamp Duty, First + Last 12 Months Amortisation,
#           Loan Comparison with HK Prime Rate (Prime - 1.75%)
# No colour codes – plain console output
# ============================================================

import math
import heapq

# ------------------------- Property Class with Real Stamp Duty -------------------------
class Property:
    # Hong Kong 2026 Stamp Duty (Scale 2) - progressive brackets
    STAMP_DUTY_BRACKETS = [
        (4_000_000, "fixed", 100),
        (4_323_780, "progressive", 100, 0.20, 4_000_000),
        (4_500_000, "percentage", 0.015),
        (4_935_480, "progressive", 67_500, 0.10, 4_500_000),
        (6_000_000, "percentage", 0.0225),
        (6_642_860, "progressive", 135_000, 0.10, 6_000_000),
        (9_000_000, "percentage", 0.03),
        (10_080_000, "progressive", 270_000, 0.10, 9_000_000),
        (20_000_000, "percentage", 0.0375),
        (21_739_120, "progressive", 750_000, 0.10, 20_000_000),
        (100_000_000, "percentage", 0.0425),
        (109_574_470, "progressive", 4_250_000, 0.30, 100_000_000),
        (float('inf'), "percentage", 0.065)
    ]
    
    def __init__(self, price):
        self.price = price
    
    def calculate_stamp_duty(self):
        price = self.price
        for limit, typ, *params in self.STAMP_DUTY_BRACKETS:
            if price <= limit:
                if typ == "fixed":
                    return params[0]
                elif typ == "percentage":
                    return price * params[0]
                elif typ == "progressive":
                    base, rate, threshold = params[0], params[1], params[2]
                    return base + (price - threshold) * rate
        return price * 0.065

# ------------------------- Base Loan Class -------------------------
class Loan:
    """Abstract base class for all loan types."""
    
    def __init__(self, principal, annual_rate, years):
        self.principal = principal
        self.annual_rate = annual_rate
        self.years = years
    
    def monthly_payment(self):
        raise NotImplementedError
    
    def total_payment(self):
        monthly = self.monthly_payment()
        return monthly * self.years * 12
    
    def total_interest(self):
        return self.total_payment() - self.principal
    
    def __lt__(self, other):
        return self.total_payment() < other.total_payment()

class FixedRateLoan(Loan):
    """Standard fixed-rate mortgage."""
    
    def monthly_payment(self):
        if self.principal <= 0 or self.years <= 0:
            return 0.0
        monthly_rate = self.annual_rate / 100 / 12
        n = self.years * 12
        if monthly_rate == 0:
            return self.principal / n
        return self.principal * monthly_rate * (1 + monthly_rate)**n / ((1 + monthly_rate)**n - 1)

# ------------------------- Loan Calculator (Static Methods) -------------------------
class LoanCalculator:
    @staticmethod
    def monthly_payment(principal, annual_rate, years):
        return FixedRateLoan(principal, annual_rate, years).monthly_payment()
    
    @staticmethod
    def repayment_period(principal, monthly_pmt, annual_rate):
        if principal <= 0 or monthly_pmt <= 0:
            return 0.0
        monthly_rate = annual_rate / 100 / 12
        if monthly_rate == 0:
            return principal / monthly_pmt / 12
        if monthly_pmt <= principal * monthly_rate:
            return float('inf')
        n = math.log(monthly_pmt / (monthly_pmt - principal * monthly_rate)) / math.log(1 + monthly_rate)
        return n / 12
    
    @staticmethod
    def interest_rate(principal, monthly_pmt, years):
        if principal <= 0 or monthly_pmt <= 0 or years <= 0:
            return None
        zero_interest_pmt = principal / (years * 12)
        if monthly_pmt < zero_interest_pmt - 1e-9:
            return None
        
        def diff(r_monthly):
            if r_monthly < 1e-12:
                return zero_interest_pmt - monthly_pmt
            n = years * 12
            pmt = principal * r_monthly * (1 + r_monthly)**n / ((1 + r_monthly)**n - 1)
            return pmt - monthly_pmt
        
        low, high = 0.0, 0.1
        for _ in range(100):
            mid = (low + high)/2
            d = diff(mid)
            if abs(d) < 1e-8:
                break
            if d > 0:
                high = mid
            else:
                low = mid
        return (low + high)/2 * 12 * 100

# ------------------------- Amortisation Table (First & Last 12 Months) -------------------------
def amortisation_table(principal, annual_rate, years, months_to_show=12):
    """Print amortisation schedule (first and last 12 months) and summary."""
    loan = FixedRateLoan(principal, annual_rate, years)
    monthly_pmt = loan.monthly_payment()
    if monthly_pmt == 0:
        print("Invalid loan parameters.")
        return
    
    remaining = principal
    monthly_rate = annual_rate / 100 / 12
    total_interest = 0
    
    total_months = int(round(years * 12))
    if total_months <= 0:
        print("Invalid loan term.")
        return
    
    # Store payment details for each month
    schedule = []
    for month in range(1, total_months + 1):
        interest = remaining * monthly_rate
        principal_part = monthly_pmt - interest
        if principal_part > remaining:
            principal_part = remaining
            monthly_pmt_adj = interest + principal_part
        else:
            monthly_pmt_adj = monthly_pmt
        remaining -= principal_part
        total_interest += interest
        schedule.append((month, monthly_pmt_adj, principal_part, interest, remaining))
        if remaining <= 0:
            break
    
    print("\n=== Amortisation Schedule ===")
    print(f"Loan Amount: HKD {principal:,.2f}, Interest Rate: {annual_rate}%, Term: {years} years")
    print(f"Monthly Payment: HKD {monthly_pmt:,.2f}\n")
    print(f"{'Month':<8} {'Payment':>12} {'Principal':>12} {'Interest':>12} {'Remaining':>14}")
    print("-" * 60)
    
    # Display first 12 months
    for i, (month, pmt, prin, int_, rem) in enumerate(schedule):
        if i >= months_to_show:
            break
        print(f"{month:<8} {pmt:>12,.2f} {prin:>12,.2f} {int_:>12,.2f} {rem:>14,.2f}")
    
    if len(schedule) > months_to_show * 2:
        print("... (skip middle months) ...")
        # Display last 12 months
        for i, (month, pmt, prin, int_, rem) in enumerate(schedule[-months_to_show:], start=len(schedule)-months_to_show+1):
            print(f"{month:<8} {pmt:>12,.2f} {prin:>12,.2f} {int_:>12,.2f} {rem:>14,.2f}")
    else:
        # Not enough months to skip, just show all remaining
        for i, (month, pmt, prin, int_, rem) in enumerate(schedule[months_to_show:], start=months_to_show+1):
            print(f"{month:<8} {pmt:>12,.2f} {prin:>12,.2f} {int_:>12,.2f} {rem:>14,.2f}")
    
    print("-" * 60)
    print(f"Total Interest Paid: HKD {total_interest:,.2f}")
    print(f"Total Payment: HKD {loan.total_payment():,.2f}")

# ------------------------- Compare Loan Options (with HK Prime Rate reference) -------------------------
# Hong Kong Prime Rate as of April 2026 (major banks) - 5.00% p.a.
HK_PRIME_RATE = 5.0
# Reference rate = Prime - 1.75% = 3.25% p.a. (common HIBOR cap rate)
REFERENCE_RATE = HK_PRIME_RATE - 1.75

def compare_two_loans(principal, option1, option2):
    """
    option1, option2: tuples (rate, years)
    Uses heap to find the better option.
    """
    loans = []
    for rate, years in [option1, option2]:
        loan = FixedRateLoan(principal, rate, years)
        monthly = loan.monthly_payment()
        total = loan.total_payment()
        interest = loan.total_interest()
        heapq.heappush(loans, (total, rate, years, monthly, interest))
    
    print("\n=== Loan Comparison ===")
    print(f"Principal: HKD {principal:,.2f}\n")
    print(f"Market Reference: HK Prime Rate = {HK_PRIME_RATE}% -> Reference Rate = {REFERENCE_RATE}% (Prime - 1.75%)")
    print()
    
    # Show both options
    for total, rate, years, monthly, interest in loans:
        print(f"Option: {rate}% for {years} years")
        print(f"  Monthly: HKD {monthly:,.2f} | Total: HKD {total:,.2f} | Interest: HKD {interest:,.2f}\n")
    
    best = loans[0]
    print(f"Best option: {best[1]}% for {best[2]} years")
    print(f"  Monthly: HKD {best[3]:,.2f} | Total: HKD {best[0]:,.2f} | Interest: HKD {best[4]:,.2f}")
    
    # Compare best option with reference rate
    ref_loan = FixedRateLoan(principal, REFERENCE_RATE, 30)  # Assume 30-year standard
    ref_monthly = ref_loan.monthly_payment()
    ref_total = ref_loan.total_payment()
    print("\nComparison with Standard Reference (Prime-1.75% for 30 years):")
    print(f"  Monthly: HKD {ref_monthly:,.2f} | Total: HKD {ref_total:,.2f}")
    if best[0] < ref_total:
        print("✓ Your best option is better than the reference rate!")
    else:
        print("✗ The reference rate may offer better value.")

# ------------------------- Main Program -------------------------
def main():
    print("\n=== Hong Kong Property & Mortgage Calculator (Advanced) ===")
    print("Features: Real 2026 Stamp Duty, Amortisation Table (First + Last 12 Months),")
    print("          Loan Comparison with HK Prime Rate (Prime - 1.75%)\n")
    
    # Property input (price in millions)
    price_millions = float(input("Enter property price (in MILLIONS HKD, e.g., 4 = 4,000,000): "))
    property_price = price_millions * 1_000_000
    print(f"Property price: HKD {property_price:,.0f}")
    
    ltv = float(input("Enter mortgage percentage (e.g., 70 for 70%): "))
    if ltv < 0 or ltv > 100:
        print("Invalid LTV, using 70%.")
        ltv = 70
    
    # Stamp duty and first mortgage
    prop = Property(property_price)
    stamp_duty = prop.calculate_stamp_duty()
    first_mortgage = property_price * (ltv / 100)
    
    print(f"\nStamp Duty payable: HKD {stamp_duty:,.2f}")
    print(f"First Mortgage Amount: HKD {first_mortgage:,.2f}")
    
    loan_amount = first_mortgage
    
    # Main menu
    while True:
        print("\n=== Main Menu ===")
        print("1. Calculate monthly payment (given period & rate)")
        print("2. Calculate repayment period (given monthly & rate)")
        print("3. Calculate interest rate (given monthly & period)")
        print("4. Show amortisation table (first + last 12 months)")
        print("5. Compare loan options (with HK Prime Rate reference)")
        print("0. Exit")
        choice = input("Enter choice: ").strip()
        
        if choice == '0':
            print("Thank you for using the calculator. Goodbye!")
            break
        
        elif choice == '1':
            years = float(input("Repayment period (years): "))
            rate = float(input("Annual interest rate (%): "))
            if years <= 0 or rate < 0:
                print("Error: Years must be positive and rate non-negative.")
                continue
            monthly = LoanCalculator.monthly_payment(loan_amount, rate, years)
            print(f"\nMonthly payment: HKD {monthly:,.2f}")
        
        elif choice == '2':
            monthly_pmt = float(input("Desired monthly payment (HKD): "))
            rate = float(input("Annual interest rate (%): "))
            if monthly_pmt <= 0 or rate < 0:
                print("Error: Monthly payment > 0 and rate >= 0.")
                continue
            years = LoanCalculator.repayment_period(loan_amount, monthly_pmt, rate)
            if math.isinf(years):
                print("\nWarning: Monthly payment too low – loan never repaid.")
            else:
                print(f"\nRepayment period: {years:.1f} years ({years*12:.0f} months)")
        
        elif choice == '3':
            monthly_pmt = float(input("Desired monthly payment (HKD): "))
            years = float(input("Repayment period (years): "))
            if monthly_pmt <= 0 or years <= 0:
                print("Error: Both must be positive.")
                continue
            rate = LoanCalculator.interest_rate(loan_amount, monthly_pmt, years)
            if rate is None:
                zero_pmt = loan_amount / (years * 12)
                print(f"\nError: Monthly payment HKD {monthly_pmt:,.2f} is below zero‑interest payment HKD {zero_pmt:,.2f}.")
            else:
                print(f"\nRequired annual interest rate: {rate:.3f}%")
        
        elif choice == '4':
            years = float(input("Loan repayment period (years): "))
            rate = float(input("Annual interest rate (%): "))
            if years <= 0 or rate < 0:
                print("Invalid input.")
                continue
            amortisation_table(loan_amount, rate, years, months_to_show=12)
        
        elif choice == '5':
            print("\n=== Loan Comparison with HK Prime Rate Reference ===")
            print(f"Current HK Prime Rate: {HK_PRIME_RATE}%")
            print(f"Reference rate (Prime - 1.75%): {REFERENCE_RATE}%")
            print("\nEnter your first loan option:")
            rate1 = float(input("  Interest rate (%): "))
            years1 = float(input("  Repayment period (years): "))
            print("\nEnter your second loan option:")
            rate2 = float(input("  Interest rate (%): "))
            years2 = float(input("  Repayment period (years): "))
            compare_two_loans(loan_amount, (rate1, years1), (rate2, years2))
        
        else:
            print("Invalid choice. Please enter 0-5.")

if __name__ == "__main__":
    main()


=== Hong Kong Property & Mortgage Calculator (Advanced) ===
Features: Real 2026 Stamp Duty, Amortisation Table (First + Last 12 Months),
          Loan Comparison with HK Prime Rate (Prime - 1.75%)



Enter property price (in MILLIONS HKD, e.g., 4 = 4,000,000):  4.5


Property price: HKD 4,500,000


Enter mortgage percentage (e.g., 70 for 70%):  80



Stamp Duty payable: HKD 67,500.00
First Mortgage Amount: HKD 3,600,000.00

=== Main Menu ===
1. Calculate monthly payment (given period & rate)
2. Calculate repayment period (given monthly & rate)
3. Calculate interest rate (given monthly & period)
4. Show amortisation table (first + last 12 months)
5. Compare loan options (with HK Prime Rate reference)
0. Exit


Enter choice:  2
Desired monthly payment (HKD):  34444
Annual interest rate (%):  2



Repayment period: 9.6 years (115 months)

=== Main Menu ===
1. Calculate monthly payment (given period & rate)
2. Calculate repayment period (given monthly & rate)
3. Calculate interest rate (given monthly & period)
4. Show amortisation table (first + last 12 months)
5. Compare loan options (with HK Prime Rate reference)
0. Exit
